# Toy Molecular Generator with LSTM
We will use a simple dataset to show case how LSTM can be used to generate SMILES strings for molecules.

Because we use such as small dataset, you might get an invalid SMILES =[

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

## We generate a tiny dataset of SMILES

In [2]:
smiles_list = [
    "C",
    "CC",
    "CCC",
    "CCCC",
    "CCO",
    "CC=O",
    "C=O",
    "C=C",
    "CC(=O)O",
    "C1CC1",
    "C1CCO1",
    "CO",
    "CCN",
    "CC(C)O",
    "CN",
    "C=NCO",
    "C=CO",
    "C1=CC=CC=C1"
]

## First, build the vocabulary
We will take each character as a token

In [3]:
def build_vocab(smiles_list):
    chars = set()
    for smi in smiles_list:
        chars.update(list(smi))
    # add special tokens
    chars = ["<PAD>", "<SOS>", "<EOS>"] + sorted(chars) # special tokens
    token_to_idx = {tok: i for i, tok in enumerate(chars)}
    idx_to_token = {i: tok for i, tok in enumerate(chars)}
    return token_to_idx, idx_to_token, len(chars)

In [5]:
token_to_idx, idx_to_token, vocab_size = build_vocab(smiles_list)
print("Vocabulary:", token_to_idx)

Vocabulary: {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '(': 3, ')': 4, '1': 5, '=': 6, 'C': 7, 'N': 8, 'O': 9}


## Next, we encode the SMILES as a sequence of tokens
Starting from `<SOS>` and ending with `<EOS>`

In [6]:
def encode_smiles(smiles_list, token_to_idx):
    sequences = []
    for smi in smiles_list:
        seq = [token_to_idx['<SOS>']] + [token_to_idx[c] for c in smi] + [token_to_idx['<EOS>']]
        sequences.append(seq)
    return sequences

In [7]:
sequence_list = encode_smiles(smiles_list, token_to_idx)
print("Encoded sequences:", sequence_list)

Encoded sequences: [[1, 7, 2], [1, 7, 7, 2], [1, 7, 7, 7, 2], [1, 7, 7, 7, 7, 2], [1, 7, 7, 9, 2], [1, 7, 7, 6, 9, 2], [1, 7, 6, 9, 2], [1, 7, 6, 7, 2], [1, 7, 7, 3, 6, 9, 4, 9, 2], [1, 7, 5, 7, 7, 5, 2], [1, 7, 5, 7, 7, 9, 5, 2], [1, 7, 9, 2], [1, 7, 7, 8, 2], [1, 7, 7, 3, 7, 4, 9, 2], [1, 7, 8, 2], [1, 7, 6, 8, 7, 9, 2], [1, 7, 6, 7, 9, 2], [1, 7, 5, 6, 7, 7, 6, 7, 7, 6, 7, 5, 2]]


### RNNs require the input to have the same length, so we need to pad the sequences

In [8]:
def pad_sequences(sequences, pad_idx):
    max_len = max(len(seq) for seq in sequences)
    padded = []
    for seq in sequences:
        padded.append(seq + [pad_idx]*(max_len - len(seq)))
    return torch.tensor(padded, dtype=torch.long)


In [10]:
X = pad_sequences(sequence_list, pad_idx=token_to_idx['<PAD>'])
Y = X[:, 1:]  # next-token targets
X = X[:, :-1] # input excludes last token
print("X shape:", X.shape, "Y shape:", Y.shape)

X shape: torch.Size([18, 12]) Y shape: torch.Size([18, 12])


## Define the LSTM model

In [11]:
class SMILES_LSTM(nn.Module):
    def __init__(self, vocab_size, embed_size=32, hidden_size=64, num_layers=1):
        super().__init__()
        # add an extra embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size) # we do not add the softmax in the model, but use it later

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out)
        return out, hidden

## Train the model

In [12]:
model = SMILES_LSTM(vocab_size)
criterion = nn.CrossEntropyLoss(ignore_index=token_to_idx["<PAD>"]) # ignore padding
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

num_epochs = 200
for epoch in range(num_epochs):
    optimizer.zero_grad()
    logits, _ = model(X)
    # reshape for cross-entropy: (batch*seq_len, vocab)
    loss = criterion(logits.reshape(-1, vocab_size), Y.reshape(-1))
    loss.backward()
    optimizer.step()

    if (epoch+1) % 50 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 50, Loss: 0.5821
Epoch 100, Loss: 0.5675
Epoch 150, Loss: 0.5672
Epoch 200, Loss: 0.5663


## Now let's generate molecules

In [13]:
def generate_smiles(model, token_to_idx, idx_to_token, max_len=50, temperature=1.0, device='cpu'):
    model.eval()
    # Start with <SOS> token
    input_idx = torch.tensor([[token_to_idx['<SOS>']]], dtype=torch.long).to(device)
    hidden = None
    generated = []

    for _ in range(max_len):
        output, hidden = model(input_idx, hidden)  # output: (1, 1, vocab_size)
        logits = output[:, -1, :] / temperature     # scale by temperature
        probs = F.softmax(logits, dim=-1)
        next_idx = torch.multinomial(probs, num_samples=1).item()

        if idx_to_token[next_idx] == '<EOS>':
            break
        generated.append(idx_to_token[next_idx])
        input_idx = torch.tensor([[next_idx]], dtype=torch.long).to(device)

    return ''.join(generated)


In [15]:
device = 'cpu'
for i in range(50):
    smiles_gen = generate_smiles(model, token_to_idx, idx_to_token, max_len=20)
    print("Generated SMILES:", smiles_gen)

Generated SMILES: CCN
Generated SMILES: CCCC
Generated SMILES: C=CO
Generated SMILES: CC(=O)O
Generated SMILES: CO
Generated SMILES: C=O
Generated SMILES: C=O
Generated SMILES: C1CCO1
Generated SMILES: C1CCO1
Generated SMILES: C1CCO1
Generated SMILES: CC(=O)O
Generated SMILES: C1CC1
Generated SMILES: CC(=O)O
Generated SMILES: C1CC1
Generated SMILES: CCCC
Generated SMILES: CC(=O)O
Generated SMILES: CC(C)O
Generated SMILES: CO
Generated SMILES: CCC
Generated SMILES: C=C
Generated SMILES: C
Generated SMILES: C=NCO
Generated SMILES: C=NCO
Generated SMILES: CCC
Generated SMILES: C1=CC=CC=C1
Generated SMILES: CN
Generated SMILES: CO
Generated SMILES: CCCC
Generated SMILES: CC=O
Generated SMILES: CC(=O)O
Generated SMILES: CCCC
Generated SMILES: CC(C)O
Generated SMILES: C=O
Generated SMILES: C)O
Generated SMILES: C
Generated SMILES: CO
Generated SMILES: C1=CC=CC=C1
Generated SMILES: CO
Generated SMILES: CCCC
Generated SMILES: C
Generated SMILES: C1CC1
Generated SMILES: CC
Generated SMILES: C=C